In [0]:
%run "../config/schema_configs"

In [0]:
from pyspark.sql.functions import col, sum, count, when, current_timestamp

print(f"🚀 Inicializando o Pipeline Analítico (Camada Gold)...")

# configura caminhos das tabelas Silver (Origens)
tabela_silver_click = SILVER_CONFIG["clickstream"]["target_table"]
tabela_silver_sales = SILVER_CONFIG["sales"]["target_table"]

# ==============================================================================
# --- MODELAGEM 1: DIMENSÃO DE COMPORTAMENTO DO USUÁRIO (dim_users_activity) ---
# ==============================================================================
print("\n📊 Construindo Dimensão: dim_users_activity...")

# lê os dados limpos da silver
df_silver_click = spark.read.table(tabela_silver_click)

# Agregação de dados de comportamento por usuário (Pivotamento manual de alta performance)
df_gold_user_activity = df_silver_click.groupBy("user_id") \
    .agg(
        count(when(col("action") == "view_product", 1)).alias("total_views"),
        count(when(col("action") == "add_to_cart", 1)).alias("total_adds_to_cart"),
        count(when(col("action") == "remove_from_cart", 1)).alias("total_removes_from_cart"),
        count(when(col("action") == "click_banner", 1)).alias("total_banner_clicks"),
        sum(when(col("discount_applied") == True, 1)).cast("long").alias("total_discounts_clicked"),
        current_timestamp().alias("gold_updated_timestamp")
    ) \
    .filter(col("user_id").isNotNull()) # Negócio só quer analisar usuários identificados

# Salva a Dimensão na Gold (Tabela de consulta rápida)
df_gold_user_activity.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(GOLD_CONFIG["dim_users_activity"])

print(f" ✅ Tabela dim_users_activity consolidada: {GOLD_CONFIG['dim_users_activity']}")    

# ==============================================================================
# --- MODELAGEM 2: TABELA FATO DE VENDAS (fct_sales) ---
# ==============================================================================
print("\n📊 Construindo Tabela Fato: fct_sales...")

# lê os dados limpos da silver
df_silver_sales = spark.read.table(tabela_silver_sales)

# Seleciona apenas as colunas chave de negócio para aplicar as regras de negócio finais
df_fct_sales = df_silver_sales.select(
    col("transaction_id").alias("sk_transaction"),
    col("user_id"),
    col("product_id"),
    col("amount").alias("vlr_venda"),
    col("payment_method").alias("desc_metodo_pagamento"),
    col("transaction_timestamp").alias("dt_transacao"),
    current_timestamp().alias("gold_ingestion_timestamp")
)

df_fct_sales.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(GOLD_CONFIG["fct_sales"])

print(f" ✅ Tabela fct_sales consolidada: {GOLD_CONFIG['fct_sales']}")

# ==============================================================================
# --- MODELAGEM 3: AGREGAÇÃO EXECUTIVA DE FATURAMENTO ---
# ==============================================================================
print("\n📊 Construindo Agregação Comercial: agg_faturamento_pagamento...")

# Agrupando a Fato recém-criada para gerar um insight rápido de negócio
df_agg_comercial = df_fct_sales.groupBy("desc_metodo_pagamento") \
    .agg(
        count("sk_transaction").alias("qtd_vendas"),
        sum("vlr_venda").alias("faturamento_total"),
        current_timestamp().alias("gold_ingestion_timestamp")
    ) \
    .orderBy(col("faturamento_total").desc())

df_agg_comercial.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(GOLD_CONFIG["agg_faturamento_pagamento"])

print(f" ✅ Tabela agg_faturamento_pagamento consolidada: {GOLD_CONFIG['agg_faturamento_pagamento']}")
print("Pipeline Analítico (Camada Gold) finalizado com sucesso!")

